# SIGNOVA HISTORICAL PIPELINE NOTICE

## KArSL MediaPipe Extraction Pilot

Historical notebook containing video QC and a MediaPipe extraction pilot limited to five videos.

### Execution safety
- Historical research code: **do not run all cells automatically**.
- Review all Google Drive paths before running individual cells.
- Some cells extract archives and create or overwrite CSV reports.
- Some cells delete temporary directories under `/content`.
- Run only in an isolated Colab runtime with the required source files.
- Confirm sufficient disk space and dependencies before processing.

### Reproducibility limitations
- The code is preserved for provenance, not certified as end-to-end reproducible.
- External datasets, original archives and model assets are not included.
- Stored notebook outputs and execution counts have been removed.
- Full MediaPipe extraction across KArSL-502 was not performed in this project.
- The external Pose dataset is a separate third-party source.
- Source licenses, label approvals and video-to-pose identities remain unresolved.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
print((R / "metadata_karsl_canonical.csv").exists())
print(len(list(Path("/content/drive/MyDrive/KArSL-502").rglob("*.7z"))))
!7z | head -2

In [ ]:
import pandas as pd, subprocess, hashlib, shutil, cv2
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
OUT = R / "video_qc_parts"; OUT.mkdir(exist_ok=True)
TMP = Path("/content/qc_tmp")
df = pd.read_csv(R / "metadata_karsl_canonical.csv", dtype={"signer": str}, low_memory=False)

def check(p):
    r = dict(decode_ok=False, last_frame_ok=False, frames=0, fps=0.0, width=0, height=0, duration_sec=0.0)
    cap = cv2.VideoCapture(str(p))
    if cap.isOpened():
        r["frames"] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        r["fps"] = float(cap.get(cv2.CAP_PROP_FPS))
        r["width"] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        r["height"] = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if r["fps"] > 0: r["duration_sec"] = round(r["frames"] / r["fps"], 3)
        r["decode_ok"] = bool(cap.read()[0])
        if r["frames"] > 1:
            cap.set(cv2.CAP_PROP_POS_FRAMES, r["frames"] - 1)
            r["last_frame_ok"] = bool(cap.read()[0])
    cap.release()
    r["file_bytes"] = p.stat().st_size
    r["sha256"] = hashlib.sha256(p.read_bytes()).hexdigest()
    return r

for arch, g in df.groupby("archive_path"):
    a = Path(arch)
    part = OUT / f"{a.parent.parent.name}_{a.parent.name}_{a.stem}.csv"
    if part.exists():
        continue
    print("Processing", part.name, len(g), "videos", flush=True)
    shutil.rmtree(TMP, ignore_errors=True); TMP.mkdir(parents=True)
    subprocess.run(["7z", "x", arch, f"-o{TMP}", "-y", "-bso0", "-bsp0"], check=True)
    rows = []
    for sid, rel in zip(g.sample_id, g.video_path_in_archive):
        p = TMP / rel
        r = check(p) if p.exists() else {"missing": True}
        r["sample_id"] = sid
        rows.append(r)
    pd.DataFrame(rows).to_csv(part, index=False)
    shutil.rmtree(TMP, ignore_errors=True)

print("Done parts:", len(list(OUT.glob("*.csv"))), "/ 30")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import glob

csv_files = glob.glob('*metadata*.csv')
if not csv_files:
    csv_files = glob.glob('/content/drive/MyDrive/**/*metadata*.csv', recursive=True)
if not csv_files:
    csv_files = glob.glob('*.csv')

df = pd.read_csv(csv_files[0])

path_col = next((c for c in df.columns if any(x in str(c).lower() for x in ['path', 'video', 'file'])), df.columns[3])
label_col = next((c for c in df.columns if any(x in str(c).lower() for x in ['sign_id', 'label', 'class'])), df.columns[4])

if 'qc_status' in df.columns:
    df = df[df['qc_status'] != 'EXCLUDE']

df = df.groupby(label_col).filter(lambda x: len(x) >= 3)

X = df[[path_col, label_col]]
y = df[label_col]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42)

df['split_assigned'] = 'unassigned'
df.loc[X_train.index, 'split_assigned'] = 'train'
df.loc[X_val.index, 'split_assigned'] = 'val'
df.loc[X_test.index, 'split_assigned'] = 'test'

df.to_csv('split_manifest.csv', index=False)
print("File:", csv_files[0])
print(df['split_assigned'].value_counts())

In [ ]:
from pathlib import Path

locations = [
    Path("/content/drive/MyDrive/KArSL-502"),
    Path("/content/drive/MyDrive/SIGNOVA/KArSL"),
    Path("/content/KArSL-502"),
    Path("/content/KArSL"),
    Path("/content")
]

extensions = {".mp4", ".avi", ".mov", ".mkv"}

print("=== DATASET LOCATION CHECK ===")

for folder in locations:
    if not folder.exists():
        print(f"{folder}: NOT FOUND")
        continue

    videos = [
        p for p in folder.rglob("*")
        if p.suffix.lower() in extensions
    ]

    print(f"\nLocation: {folder}")
    print(f"Videos found: {len(videos)}")

    for video in videos[:3]:
        print("Sample:", video)

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive")
reports = root / "SIGNOVA/KArSL/reports"
m = pd.read_csv(reports / "metadata_arsl.csv", nrows=3)

print("=== METADATA PATHS ===")
for col in m.columns:
    if "path" in col.lower() or "file" in col.lower():
        print(col, ":", m[col].tolist())

print("\n=== GOOGLE DRIVE FOLDERS ===")
for p in root.iterdir():
    if p.is_dir():
        print(p.name)

print("\n=== DATASET STORAGE ===")
for folder in [root / "KArSL-502", root / "SIGNOVA/KArSL"]:
    if folder.exists():
        print("\nFolder:", folder)
        for p in list(folder.iterdir())[:50]:
            print("DIR " if p.is_dir() else "FILE", p.name)

print("\n=== ARCHIVES ===")
for folder in [root, root / "KArSL-502", root / "SIGNOVA/KArSL"]:
    if folder.exists():
        for p in folder.iterdir():
            if p.suffix.lower() in [".7z", ".zip", ".rar"]:
                print(p)

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

root = Path("/content/drive/MyDrive/KArSL-502")
m = pd.read_csv("/content/drive/MyDrive/SIGNOVA/KArSL/reports/metadata_arsl.csv")

archives = sorted(root.rglob("*.7z"))
expected = set(m["archive_relpath"].dropna().astype(str))
found = {str(p.relative_to(root)) for p in archives}

print("=== ARCHIVE INVENTORY ===")
print("Expected archives:", len(expected))
print("Found archives:", len(archives))
print("Missing archives:", len(expected - found))
print("Total size GB:", round(sum(p.stat().st_size for p in archives) / 1024**3, 2))

print("\n=== ARCHIVE LOCATIONS ===")
for p in archives:
    print(p.relative_to(root))

print("\n=== MISSING ARCHIVES ===")
for p in sorted(expected - found):
    print(p)

print("\n=== ARCHIVE VIDEO COUNTS ===")
print(m.groupby("archive_relpath").size().to_string())

In [ ]:
import sys
import subprocess
import importlib.metadata

print("Python version:", sys.version.split()[0])

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.30"],
    capture_output=True,
    text=True
)

print("Installation:", "SUCCESS" if result.returncode == 0 else "FAILED")

if result.returncode == 0:
    import mediapipe as mp
    print("MediaPipe version:", importlib.metadata.version("mediapipe"))
    print("Legacy Holistic available:", hasattr(mp, "solutions") and hasattr(mp.solutions, "holistic"))
    print("Tasks API available:", hasattr(mp, "tasks"))
else:
    print(result.stderr[-1500:])

In [ ]:
from pathlib import Path
import urllib.request
import mediapipe as mp

MODELS = Path("/content/drive/MyDrive/SIGNOVA/KArSL/models")
MODELS.mkdir(parents=True, exist_ok=True)

urls = {
    "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
    "pose_landmarker.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
}

print("=== MODEL DOWNLOAD ===")

for name, url in urls.items():
    path = MODELS / name
    try:
        if not path.exists() or path.stat().st_size == 0:
            urllib.request.urlretrieve(url, path)
        print(name, "READY", round(path.stat().st_size / 1048576, 2), "MB")
    except Exception as e:
        print(name, "FAILED", str(e)[:200])

print("\n=== MODEL INITIALIZATION ===")

vision = mp.tasks.vision
classes = {
    "Hand": (vision.HandLandmarker, vision.HandLandmarkerOptions, "hand_landmarker.task"),
    "Pose": (vision.PoseLandmarker, vision.PoseLandmarkerOptions, "pose_landmarker.task"),
    "Face": (vision.FaceLandmarker, vision.FaceLandmarkerOptions, "face_landmarker.task")
}

for name, (cls, options, filename) in classes.items():
    try:
        with cls.create_from_options(
            options(base_options=mp.tasks.BaseOptions(
                model_asset_path=str(MODELS / filename)
            ))
        ):
            print(name, "PASS")
    except Exception as e:
        print(name, "FAIL", str(e)[:200])

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import subprocess
import shutil
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/KArSL-502")
BASE = Path("/content/drive/MyDrive/SIGNOVA/KArSL")
R = BASE / "reports"
MODELS = BASE / "models"
OUT = BASE / "keypoints/pilot"
TMP = Path("/content/karsl_pilot")

OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["apt-get", "-qq", "install", "-y", "p7zip-full"],
    check=True
)

m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv", low_memory=False)

archive_name = "01/test/0171-0190.7z"
valid = set(s.sample_id.astype(str))

samples = m[
    (m.archive_relpath == archive_name) &
    (m.sample_id.astype(str).isin(valid))
].head(5)

if len(samples) != 5:
    raise ValueError("Five valid samples were not found")

paths = samples.video_path_in_archive.tolist()

print("=== ARCHIVE EXTRACTION ===")

cmd = [
    "7z", "x", str(ROOT / archive_name),
    f"-o{TMP}", "-y", *paths
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout[-1000:])
    print(result.stderr[-1000:])
    raise RuntimeError("Archive extraction failed")

print("Archive extraction: PASS")

vision = mp.tasks.vision
Base = mp.tasks.BaseOptions

hand = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=Base(model_asset_path=str(MODELS / "hand_landmarker.task")),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2
    )
)

pose = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=Base(model_asset_path=str(MODELS / "pose_landmarker.task")),
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1
    )
)

face = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=Base(model_asset_path=str(MODELS / "face_landmarker.task")),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1
    )
)

def landmarks(points, count, dims=3):
    a = np.zeros((count, dims), dtype=np.float32)
    if points:
        for i, p in enumerate(points[:count]):
            a[i, :3] = [p.x, p.y, p.z]
            if dims == 4:
                a[i, 3] = getattr(p, "visibility", 0)
    return a.flatten()

def process_frame(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=np.ascontiguousarray(rgb)
    )

    h = hand.detect(image)
    p = pose.detect(image)
    f = face.detect(image)

    hands = {
        "Left": np.zeros(63, dtype=np.float32),
        "Right": np.zeros(63, dtype=np.float32)
    }

    for points, labels in zip(h.hand_landmarks, h.handedness):
        side = labels[0].category_name
        if side in hands:
            hands[side] = landmarks(points, 21)

    pose_data = landmarks(
        p.pose_landmarks[0] if p.pose_landmarks else None, 33, 4
    )

    face_data = landmarks(
        f.face_landmarks[0] if f.face_landmarks else None, 478
    )

    data = np.concatenate([
        pose_data,
        face_data,
        hands["Left"],
        hands["Right"]
    ])

    detected = {
        "pose": bool(p.pose_landmarks),
        "face": bool(f.face_landmarks),
        "hands": len(h.hand_landmarks)
    }

    return data, detected

records = []

print("\n=== MEDIAPIPE EXTRACTION ===")

try:
    for _, row in samples.iterrows():
        video = TMP / row.video_path_in_archive
        sample_id = str(row.sample_id)

        if not video.exists():
            records.append({
                "sample_id": sample_id,
                "status": "MISSING_VIDEO"
            })
            continue

        cap = cv2.VideoCapture(str(video))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total <= 0:
            cap.release()
            records.append({
                "sample_id": sample_id,
                "status": "INVALID_VIDEO"
            })
            continue

        indices = np.unique(
            np.linspace(0, total - 1, min(32, total)).astype(int)
        )

        sequence = []
        detections = []

        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()

            if not ok:
                continue

            data, detected = process_frame(frame)
            sequence.append(data)
            detections.append(detected)

        cap.release()

        if sequence:
            arr = np.stack(sequence)
            np.save(OUT / f"{sample_id}.npy", arr)

            record = {
                "sample_id": sample_id,
                "frames": arr.shape[0],
                "features": arr.shape[1],
                "pose_rate": round(np.mean([x["pose"] for x in detections]), 3),
                "face_rate": round(np.mean([x["face"] for x in detections]), 3),
                "hand_rate": round(np.mean([x["hands"] > 0 for x in detections]), 3),
                "status": "SUCCESS"
            }
        else:
            record = {
                "sample_id": sample_id,
                "status": "FAILED"
            }

        records.append(record)
        print(sample_id, record["status"])

finally:
    hand.close()
    pose.close()
    face.close()

report = pd.DataFrame(records)
report.to_csv(OUT / "pilot_report.csv", index=False)

print("\n=== PILOT RESULTS ===")
print(report.to_string(index=False))

print("\n=== SUMMARY ===")
print("Successful:", (report.status == "SUCCESS").sum())
print("Failed:", (report.status != "SUCCESS").sum())
print("Saved NPY files:", len(list(OUT.glob("*.npy"))))
print("Output directory:", OUT)

if (report.status == "SUCCESS").all():
    shutil.rmtree(TMP)

In [ ]:
!pip -q install kagglehub

import kagglehub
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

destination = Path("/content/drive/MyDrive/SIGNOVA/KArSL/keypoints/kaggle")
destination.mkdir(parents=True, exist_ok=True)

print("=== DOWNLOADING DATASET ===")

path = kagglehub.dataset_download(
    "yousefdotpy/blablabla",
    output_dir=str(destination)
)

print("\n=== DOWNLOAD RESULTS ===")
print("Dataset location:", path)
print("Files downloaded:", sum(1 for p in destination.rglob("*") if p.is_file()))
print("Total size GB:", round(
    sum(p.stat().st_size for p in destination.rglob("*") if p.is_file()) / 1024**3, 2
))
print("Download status: COMPLETE")

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/SIGNOVA/KArSL")
K = BASE / "keypoints/kaggle"
R = BASE / "reports"

files = list(K.rglob("*.npy"))
groups = Counter(p.parent.name for p in files)

print("=== KEYPOINT INVENTORY ===")
print("Total NPY files:", len(files))
print("Feature groups:", dict(groups))

print("\n=== SAMPLE PATHS AND SHAPES ===")
for p in files[:9]:
    try:
        arr = np.load(p, mmap_mode="r")
        print(p.relative_to(K), "Shape:", arr.shape)
    except Exception as e:
        print(p.relative_to(K), "ERROR:", str(e)[:100])

samples = {}

for p in files:
    group = p.parent.name
    key = str(p.parent.parent.relative_to(K) / p.stem)
    samples.setdefault(key, set()).add(group)

coverage = Counter(tuple(sorted(v)) for v in samples.values())

print("\n=== UNIQUE SAMPLE COVERAGE ===")
print("Unique samples:", len(samples))
for combination, count in coverage.most_common():
    print("+".join(combination), count)

print("\n=== METADATA SAMPLE ===")
m = pd.read_csv(R / "metadata_arsl.csv", nrows=3, low_memory=False)

print(m[[
    "sample_id",
    "archive_relpath",
    "video_path_in_archive"
]].to_string(index=False))

print("\n=== FINAL STATUS ===")
print("Dataset downloaded: YES")
print("Keypoint inventory completed: YES")

In [ ]:
from pathlib import Path
import pandas as pd

B = Path("/content/drive/MyDrive/SIGNOVA/KArSL")
K = B / "keypoints/kaggle"
R = B / "reports"

m = pd.read_csv(R / "metadata_arsl.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv", low_memory=False)

m["sample_id"] = m["sample_id"].astype(str)
m["video_key"] = (
    m["archive_relpath"].str.split("/").str[:2].str.join("/")
    + "/"
    + m["video_path_in_archive"].str.replace(".mp4", "", regex=False)
)

records = []

for p in K.rglob("*.npy"):
    if p.parent.name != "lh_keypoints":
        continue

    rel = p.relative_to(K)
    parts = rel.parts

    if parts[0] == "karsl-502":
        parts = parts[1:]

    key = "/".join(parts[:3]) + "/" + p.stem

    records.append({
        "keypoint_key": key,
        "lh_path": str(p),
        "pose_path": str(p.parent.parent / "pose_keypoints" / p.name),
        "rh_path": str(p.parent.parent / "rh_keypoints" / p.name)
    })

k = pd.DataFrame(records)

k["complete"] = k.apply(
    lambda x: Path(x.pose_path).exists() and Path(x.rh_path).exists(),
    axis=1
)

matched = m.merge(
    k,
    left_on="video_key",
    right_on="keypoint_key",
    how="left",
    indicator=True
)

benchmark_ids = set(s.sample_id.astype(str))
matched["in_benchmark"] = matched.sample_id.isin(benchmark_ids)

report = matched[
    matched.in_benchmark & matched.complete.fillna(False)
].copy()

report.to_csv(R / "keypoints_manifest.csv", index=False)

print("=== KEYPOINT MATCHING RESULTS ===")
print("Original metadata samples:", len(m))
print("Benchmark samples:", len(benchmark_ids))
print("Kaggle keypoint samples:", len(k))
print("Complete keypoint samples:", k.complete.sum())
print("Matched metadata samples:", (matched._merge == "both").sum())
print("Matched benchmark samples:", len(report))
print("Benchmark coverage:", round(len(report) / len(benchmark_ids) * 100, 2), "%")

print("\n=== UNMATCHED KEYPOINTS ===")
print("Unmatched Kaggle samples:",
      len(set(k.keypoint_key) - set(m.video_key)))

print("\n=== SAMPLE MATCHES ===")
print(report[[
    "sample_id", "class_label", "signer_id", "lh_path"
]].head(5).to_string(index=False))

print("\nManifest saved:", R / "keypoints_manifest.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

k = pd.read_csv(R / "keypoints_manifest.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv", low_memory=False)

print("=== DATASET COVERAGE ===")
print("Samples:", k.sample_id.nunique())
print("Classes:", k.class_label.nunique())
print("Signers:", k.signer_id.nunique())
print("\nSigner distribution:")
print(k.signer_id.value_counts().to_string())

print("\n=== CLASS COVERAGE ===")
counts = k.groupby("class_label").size()
print("Classes with keypoints:", len(counts))
print("Samples per class (min):", counts.min())
print("Samples per class (max):", counts.max())
print("Classes with fewer than 10 samples:", (counts < 10).sum())

print("\n=== SPLIT COVERAGE ===")
joined = k[["sample_id"]].merge(
    s[["sample_id", "fold", "split"]],
    on="sample_id",
    how="left"
)
print(pd.crosstab(joined.fold, joined.split).to_string())

print("\n=== PATH CONSISTENCY ===")
filename_prefix = k.lh_path.map(lambda p: Path(p).stem.split("_")[0])
folder_prefix = k.lh_path.map(lambda p: Path(p).parts[-6])
mismatch = filename_prefix.ne(folder_prefix)
print("Filename/folder prefix mismatches:", mismatch.sum())
print(k.loc[mismatch, ["sample_id", "signer_id", "lh_path"]].head(3).to_string(index=False))

print("\n=== KEYPOINT QUALITY SAMPLE ===")
rows = []

for _, row in k.sample(n=min(100, len(k)), random_state=42).iterrows():
    for group in ["lh", "pose", "rh"]:
        try:
            a = np.load(row[f"{group}_path"], mmap_mode="r", allow_pickle=False)
            rows.append({
                "group": group,
                "shape": str(a.shape),
                "finite": bool(np.isfinite(a).all()),
                "all_zero": bool(np.all(a == 0)),
                "status": "PASS"
            })
        except Exception:
            rows.append({
                "group": group,
                "shape": "UNKNOWN",
                "finite": False,
                "all_zero": False,
                "status": "FAIL"
            })

q = pd.DataFrame(rows)

print("Files checked:", len(q))
print("Failed files:", (q.status == "FAIL").sum())
print("Non-finite files:", (~q.finite).sum())
print("All-zero files:", q.all_zero.sum())
print("\nShapes by group:")
print(q.groupby(["group", "shape"]).size().head(20).to_string())

q.to_csv(R / "keypoints_quality_sample.csv", index=False)

print("\n=== FINAL STATUS ===")
print("Quality sample report: SAVED")

In [ ]:
from pathlib import Path
import pandas as pd

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
k = pd.read_csv(R / "keypoints_manifest.csv", low_memory=False)

def check_path(row):
    p = Path(row.lh_path)
    folder_signer = p.parts[-5]
    folder_class = p.parts[-3]
    filename = p.stem.split("_")
    return (
        filename[1] == folder_signer
        and filename[2] == folder_class
        and row.signer_id == f"S{folder_signer}"
        and int(row.class_label) == int(folder_class)
    )

k["path_valid"] = k.apply(check_path, axis=1)

coverage = pd.crosstab(k.class_label, k.signer_id)
complete = coverage[(coverage > 0).all(axis=1)]
incomplete = coverage[(coverage == 0).any(axis=1)]

print("=== PATH VALIDATION ===")
print("Valid paths:", k.path_valid.sum())
print("Invalid paths:", (~k.path_valid).sum())

print("\n=== CLASS COVERAGE ===")
print("Total classes:", len(coverage))
print("Classes present in all 3 signers:", len(complete))
print("Classes missing at least one signer:", len(incomplete))

print("\n=== SIGNER DISTRIBUTION ===")
print(k.signer_id.value_counts().to_string())

print("\n=== PER-CLASS SAMPLE COUNTS ===")
print("Minimum per signer:", complete.min().min() if len(complete) else 0)
print("Maximum per signer:", complete.max().max() if len(complete) else 0)

print("\n=== INCOMPLETE CLASSES ===")
print(incomplete.to_string() if len(incomplete) else "None")

valid_classes = set(complete.index)
ready = k[k.class_label.isin(valid_classes) & k.path_valid].copy()

ready.to_csv(R / "keypoints_training_manifest.csv", index=False)
coverage.to_csv(R / "keypoints_class_coverage.csv")

print("\n=== FINAL RESULTS ===")
print("Training-ready candidate samples:", len(ready))
print("Training-ready candidate classes:", ready.class_label.nunique())
print("Training-ready candidate signers:", ready.signer_id.nunique())
print("Manifest saved: YES")
print("Full keypoint quality validation: PENDING")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")

k = pd.read_csv(R / "keypoints_training_manifest.csv", low_memory=False)
s = pd.read_csv(R / "split_manifest_3way.csv", low_memory=False)

columns = ["sample_id", "fold", "split", "split_version"]
splits = k.merge(s[columns], on="sample_id", how="left")

checks = {
    "All samples matched": splits.split.notna().all(),
    "62 classes": k.class_label.nunique() == 62,
    "Three signers": k.signer_id.nunique() == 3,
    "Three folds": splits.fold.nunique() == 3,
    "No duplicate IDs within folds": not splits.duplicated(
        ["fold", "sample_id"]
    ).any(),
    "Signer-independent splits": all(
        g.groupby("signer_id").split.nunique().max() == 1
        for _, g in splits.groupby("fold")
    ),
    "All classes in every split": all(
        g.class_label.nunique() == 62
        for _, g in splits.groupby(["fold", "split"])
    )
}

print("=== KEYPOINT SPLIT VALIDATION ===")
for name, result in checks.items():
    print(name, "PASS" if result else "FAIL")

if not all(checks.values()):
    raise ValueError("Split validation failed")

splits.to_csv(R / "keypoints_split_manifest.csv", index=False)

print("\n=== SPLIT DISTRIBUTION ===")
print(
    splits.groupby(["fold", "split"])
    .agg(samples=("sample_id", "count"),
         classes=("class_label", "nunique"))
    .to_string()
)

print("\n=== KEYPOINT QUALITY CHECK ===")

rows = []

for _, row in k.sample(n=min(100, len(k)), random_state=42).iterrows():
    arrays = {}
    status = "PASS"

    for group in ["lh", "pose", "rh"]:
        try:
            a = np.load(row[f"{group}_path"], allow_pickle=False)
            arrays[group] = a

            if a.ndim != 2 or a.shape[1] != 63:
                status = "INVALID_SHAPE"
            elif not np.isfinite(a).all():
                status = "NON_FINITE"
        except Exception:
            status = "READ_ERROR"

    lengths = [a.shape[0] for a in arrays.values()]

    if status == "PASS" and (not lengths or min(lengths) == 0):
        status = "EMPTY_SEQUENCE"
    elif status == "PASS" and len(set(lengths)) != 1:
        status = "FRAME_MISMATCH"

    rows.append({
        "sample_id": row.sample_id,
        "status": status,
        "frames": lengths[0] if lengths else 0,
        "zero_groups": sum(np.all(a == 0) for a in arrays.values())
    })

q = pd.DataFrame(rows)
q.to_csv(R / "keypoints_quality_v2_sample.csv", index=False)

print("Samples checked:", len(q))
print("Status distribution:")
print(q.status.value_counts().to_string())
print("Samples with all-zero groups:", (q.zero_groups > 0).sum())
print("Minimum frames:", q.frames.min())
print("Maximum frames:", q.frames.max())

print("\n=== FINAL STATUS ===")
print("Split manifest: SAVED")
print("Quality sample report: SAVED")
print("Full quality audit: PENDING")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
k = pd.read_csv(R / "keypoints_training_manifest.csv")

rows = []

for _, row in k.sample(n=min(100, len(k)), random_state=42).iterrows():
    arrays = {}
    status = "PASS"

    for group in ["lh", "pose", "rh"]:
        try:
            a = np.load(row[f"{group}_path"], allow_pickle=False)
            arrays[group] = a

            if a.ndim != 2 or a.shape[0] == 0:
                status = "INVALID_SHAPE"
            elif not np.isfinite(a).all():
                status = "NON_FINITE"
        except Exception:
            status = "READ_ERROR"

    lengths = [a.shape[0] for a in arrays.values()]

    if status == "PASS" and len(set(lengths)) != 1:
        status = "FRAME_MISMATCH"

    rows.append({
        "sample_id": row.sample_id,
        "status": status,
        "lh_shape": str(arrays["lh"].shape) if "lh" in arrays else "ERROR",
        "pose_shape": str(arrays["pose"].shape) if "pose" in arrays else "ERROR",
        "rh_shape": str(arrays["rh"].shape) if "rh" in arrays else "ERROR",
        "zero_groups": sum(np.all(a == 0) for a in arrays.values())
    })

q = pd.DataFrame(rows)
q.to_csv(R / "keypoints_quality_corrected.csv", index=False)

print("=== CORRECTED QUALITY CHECK ===")
print(q.status.value_counts().to_string())

print("\n=== FEATURE DIMENSIONS ===")
for group in ["lh", "pose", "rh"]:
    print(group)
    print(q[f"{group}_shape"].str.extract(r"\(\d+,\s*(\d+)\)")[0].value_counts().to_string())

print("\n=== ADDITIONAL RESULTS ===")
print("Files checked:", len(q) * 3)
print("Samples with zero groups:", (q.zero_groups > 0).sum())
print("Samples with frame mismatch:", (q.status == "FRAME_MISMATCH").sum())

print("\n=== SAMPLE SHAPES ===")
print(q[["lh_shape", "pose_shape", "rh_shape"]].head(5).to_string(index=False))

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import time

R = Path("/content/drive/MyDrive/SIGNOVA/KArSL/reports")
k = pd.read_csv(R / "keypoints_training_manifest.csv", low_memory=False)

expected = {"lh": 63, "pose": 99, "rh": 63}
rows = []
start = time.time()

print("=== FULL KEYPOINT QUALITY AUDIT ===", flush=True)
print("Samples to check:", len(k), flush=True)

for i, row in enumerate(k.itertuples(index=False), 1):
    record = {"sample_id": row.sample_id}
    lengths = []
    issues = []
    zero_groups = []

    for group, features in expected.items():
        try:
            a = np.load(getattr(row, f"{group}_path"), allow_pickle=False)
            record[f"{group}_frames"] = a.shape[0] if a.ndim else 0

            if a.ndim != 2 or a.shape[1] != features or a.shape[0] == 0:
                issues.append(f"{group}_INVALID_SHAPE")
            else:
                lengths.append(a.shape[0])

            if not np.isfinite(a).all():
                issues.append(f"{group}_NON_FINITE")

            if np.all(a == 0):
                zero_groups.append(group)

        except Exception:
            issues.append(f"{group}_READ_ERROR")
            record[f"{group}_frames"] = 0

    if len(lengths) == 3 and len(set(lengths)) != 1:
        issues.append("FRAME_MISMATCH")

    record["status"] = "PASS" if not issues else "FAIL"
    record["issues"] = ";".join(issues)
    record["zero_groups"] = ";".join(zero_groups)
    record["zero_group_count"] = len(zero_groups)
    rows.append(record)

    if i % 1000 == 0:
        print(f"Checked: {i}/{len(k)}", flush=True)

q = pd.DataFrame(rows)
q.to_csv(R / "keypoints_quality_full.csv", index=False)

print("\n=== QUALITY RESULTS ===")
print("Samples checked:", len(q))
print("Passed:", (q.status == "PASS").sum())
print("Failed:", (q.status == "FAIL").sum())
print("Samples with zero groups:", (q.zero_group_count > 0).sum())
print("Samples with all three groups zero:", (q.zero_group_count == 3).sum())

print("\n=== ISSUE DISTRIBUTION ===")
issues = Counter(
    issue
    for value in q.issues
    for issue in str(value).split(";")
    if issue
)
print(dict(issues) if issues else "No structural issues")

print("\n=== FRAME STATISTICS ===")
print("Minimum frames:", q.lh_frames.min())
print("Maximum frames:", q.lh_frames.max())
print("Average frames:", round(q.lh_frames.mean(), 2))

print("\n=== FINAL STATUS ===")
print("Quality report saved: YES")
print("Structural validation:", "PASS" if q.status.eq("PASS").all() else "FAIL")
print("Zero-group review:", "REQUIRED" if q.zero_group_count.gt(0).any() else "NOT REQUIRED")
print("Execution minutes:", round((time.time() - start) / 60, 2))

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

B = Path("/content/drive/MyDrive/SIGNOVA/KArSL")
R = B / "reports"
K = B / "keypoints/kaggle"
P = B / "github_upload/karsl"

k = pd.read_csv(R / "keypoints_training_manifest.csv", low_memory=False)
s = pd.read_csv(R / "keypoints_split_manifest.csv", low_memory=False)
q = pd.read_csv(R / "keypoints_quality_full.csv", low_memory=False)

paths = ["lh_path", "pose_path", "rh_path"]

for col in paths:
    k[col] = k[col].map(lambda x: str(Path(x).relative_to(K)))

manifest = s.drop(columns=paths, errors="ignore").merge(
    k[["sample_id"] + paths],
    on="sample_id",
    how="left",
    validate="many_to_one"
)

quality = q[["sample_id", "status", "zero_groups", "zero_group_count"]]

manifest = manifest.merge(
    quality,
    on="sample_id",
    how="left",
    validate="many_to_one"
)

assert len(manifest) == 8793 * 3
assert manifest[paths].notna().all().all()
assert manifest.status.eq("PASS").all()

files = {
    "data/metadata/keypoints_manifest.csv": manifest,
    "data/reports/keypoints_quality.csv": q,
    "data/reports/keypoints_class_coverage.csv": pd.read_csv(
        R / "keypoints_class_coverage.csv"
    ),
    "data/splits/keypoints_split_distribution.csv": manifest.groupby(
        ["fold", "split"]
    ).agg(
        samples=("sample_id", "size"),
        classes=("class_label", "nunique")
    ).reset_index()
}

for name, df in files.items():
    target = P / name
    target.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(target, index=False)

readme = """# KArSL-502 Dataset Engineering

## Raw Video Benchmark
- 75,095 PASS samples
- 500 classes
- 3 signers
- Three signer-independent evaluation folds

## Available Keypoints
- Source: Kaggle, yousefdotpy/blablabla
- 13,761 matched samples
- 112 covered classes
- 62 classes represented by all three signers
- 8,793 samples in the common-class benchmark
- Left Hand: 63 features
- Pose: 99 features
- Right Hand: 63 features
- Total: 225 features per frame
- Face landmarks: Not available
- Variable sequence lengths

## Quality Validation
- 8,793 samples structurally validated
- 0 structural failures
- 1,280 samples contain at least one all-zero group
- All-zero groups require interpretation and review

## Evaluation
Three signer-independent folds with separate training,
validation and test partitions.
Each training fold contains only one signer.
Model generalization is limited by signer diversity.

## Storage
The NPY files are stored separately on Google Drive.
Paths in the keypoints manifest are relative to
the Kaggle keypoints dataset root.

## Usage Restrictions
Original source license: Unresolved
Kaggle keypoints license: Unknown
Training and redistribution rights: Not verified
Expert label validation: Pending
Project-lead approval: Pending

## Status
Structural checks completed.
Not approved for unrestricted training or redistribution.
"""

docs = P / "README.md"
docs.parent.mkdir(parents=True, exist_ok=True)
docs.write_text(readme, encoding="utf-8")

print("=== GITHUB PREPARATION ===")
for p in sorted(P.rglob("*")):
    if p.is_file():
        print(p.relative_to(P))

print("\n=== FINAL SUMMARY ===")
print("Keypoint benchmark samples:", k.sample_id.nunique())
print("Classes:", k.class_label.nunique())
print("Quality passed:", q.status.eq("PASS").sum())
print("Zero-group review cases:", q.zero_group_count.gt(0).sum())
print("Relative paths:", "PASS")
print("GitHub staging directory:", P)
print("GitHub upload status: NOT UPLOADED")

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

B = Path("/content/drive/MyDrive/SIGNOVA/KArSL")
R = B / "reports"
P = B / "github_upload/karsl"

files = {
    "dataset_sources.csv": "data/metadata/dataset_sources.csv",
    "download_manifest.csv": "data/metadata/download_manifest.csv",
    "metadata_arsl.csv": "data/metadata/karsl_metadata.csv",
    "class_mapping_v3.csv": "data/metadata/karsl_class_mapping.csv",
    "qc_flags.csv": "data/reports/karsl_qc_flags.csv",
    "qc_report.md": "data/reports/karsl_qc_report.md",
    "dataset_tests_v3.csv": "data/reports/karsl_dataset_tests.csv",
    "open_review_cases.csv": "data/reports/karsl_open_review_cases.csv",
    "benchmark_v3_summary.md": "data/reports/karsl_benchmark_summary.md",
    "split_manifest_3way.csv": "data/splits/karsl_split_manifest.csv",
    "split_distribution.csv": "data/splits/karsl_split_distribution.csv",
    "split_class_gaps.csv": "data/splits/karsl_class_gaps.csv",
    "split_excluded_classes.csv": "data/splits/karsl_excluded_classes.csv",
    "dataset_card.md": "docs/karsl_dataset_card.md",
    "handoff.md": "docs/karsl_handoff.md"
}

results = []

for source, target in files.items():
    src = R / source
    dst = P / target

    if not src.exists():
        status = "MISSING"
        size = 0
    else:
        size = round(src.stat().st_size / 1048576, 2)

        if src.stat().st_size > 90 * 1048576:
            status = "TOO_LARGE"
        elif dst.exists():
            status = "ALREADY_EXISTS"
        else:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            status = "READY"

    results.append({
        "file": target,
        "size_mb": size,
        "status": status
    })

report = pd.DataFrame(results)

print("=== RAW DATASET FILES ===")
print(report.to_string(index=False))

print("\n=== GITHUB PREPARATION SUMMARY ===")
print("Files ready:", report.status.isin(["READY", "ALREADY_EXISTS"]).sum())
print("Missing files:", report.status.eq("MISSING").sum())
print("Oversized files:", report.status.eq("TOO_LARGE").sum())

print("\n=== PATH PRIVACY CHECK ===")
for name in ["data/metadata/karsl_metadata.csv",
             "data/metadata/keypoints_manifest.csv"]:
    p = P / name
    if p.exists():
        text = p.read_text(encoding="utf-8", errors="replace")
        found = any(x in text for x in [
            "/content/drive/MyDrive/",
            "C:\\Users\\",
            "/home/"
        ])
        print(name, "REVIEW_REQUIRED" if found else "PASS")

print("\n=== TOTAL PACKAGE SIZE ===")
size = sum(p.stat().st_size for p in P.rglob("*") if p.is_file())
print("Total MB:", round(size / 1048576, 2))
print("GitHub upload status: NOT UPLOADED")

In [ ]:
from pathlib import Path
import pandas as pd

P = Path("/content/drive/MyDrive/SIGNOVA/KArSL/github_upload/karsl")

files = sorted(p for p in P.rglob("*") if p.is_file())

patterns = [
    b"/content/drive/MyDrive/",
    b"C:\\Users\\",
    b"/home/",
    b"api_key",
    b"private_key",
    b"password="
]

rows = []

for p in files:
    size = p.stat().st_size
    content = p.read_bytes()
    found = [
        x.decode() for x in patterns
        if x.lower() in content.lower()
    ]

    rows.append({
        "file": str(p.relative_to(P)),
        "size_mb": round(size / 1048576, 2),
        "size_status": "REVIEW" if size > 50 * 1048576 else "PASS",
        "content_status": "REVIEW" if found else "PASS",
        "flags": ", ".join(found)
    })

report = pd.DataFrame(rows)

print("=== GITHUB FILE AUDIT ===")
print(report.to_string(index=False))

print("\n=== AUDIT SUMMARY ===")
print("Total files:", len(report))
print("Total size MB:", round(sum(p.stat().st_size for p in files) / 1048576, 2))
print("Oversized files:", report.size_status.eq("REVIEW").sum())
print("Files requiring content review:", report.content_status.eq("REVIEW").sum())

print("\n=== NOTEBOOKS CURRENTLY STAGED ===")
notebooks = list(P.rglob("*.ipynb"))
print("Notebook count:", len(notebooks))

print("\n=== FINAL STATUS ===")
print("Ready for notebook integration:",
      "YES" if report.size_status.eq("PASS").all()
      and report.content_status.eq("PASS").all() else "REVIEW_REQUIRED")

In [ ]:
from pathlib import Path
import pandas as pd

P = Path("/content/drive/MyDrive/SIGNOVA/KArSL/github_upload/karsl")
f = P / "data/metadata/download_manifest.csv"

df = pd.read_csv(f, low_memory=False)

print("=== ORIGINAL PATH COLUMNS ===")
for col in df.columns:
    if df[col].astype(str).str.contains(
        "/content/drive/MyDrive/", regex=False
    ).any():
        print(col)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.replace(
        "/content/drive/MyDrive/",
        "DRIVE_ROOT/",
        regex=False
    )

df.to_csv(f, index=False)

remaining = "/content/drive/MyDrive/" in f.read_text(
    encoding="utf-8"
)

print("\n=== CLEANUP RESULTS ===")
print("Rows:", len(df))
print("Personal paths remaining:", remaining)
print("Status:", "PASS" if not remaining else "FAIL")
print("Original reports modified: NO")

In [ ]:
from pathlib import Path
import json
import shutil

D = Path("/content/drive/MyDrive")
P = D / "SIGNOVA/KArSL/github_upload/karsl/notebooks"

names = [
    "karsl.ipynb",
    "karsl_label_review.ipynb",
    "03_MediaPipe_Extraction.ipynb"
]

P.mkdir(parents=True, exist_ok=True)

print("=== NOTEBOOK DISCOVERY ===")

for name in names:
    matches = list(D.rglob(name))
    matches = [p for p in matches if "github_upload" not in p.parts]

    if not matches:
        print(name, "NOT FOUND")
        continue

    src = max(matches, key=lambda p: p.stat().st_mtime)
    dst = P / name

    try:
        with open(src, encoding="utf-8") as f:
            nb = json.load(f)

        for cell in nb.get("cells", []):
            if cell.get("cell_type") == "code":
                cell["outputs"] = []
                cell["execution_count"] = None

        dst.write_text(
            json.dumps(nb, ensure_ascii=False, indent=1),
            encoding="utf-8"
        )

        print(name, "PREPARED")
        print("Source:", src)
        print("Cells:", len(nb.get("cells", [])))
        print("Size MB:", round(dst.stat().st_size / 1048576, 2))

    except Exception as e:
        print(name, "FAILED", str(e)[:150])

print("\n=== STAGED NOTEBOOKS ===")
for p in sorted(P.glob("*.ipynb")):
    print(p.name, round(p.stat().st_size / 1048576, 2), "MB")

In [ ]:
from pathlib import Path

D = Path("/content/drive/MyDrive")
P = D / "SIGNOVA/KArSL/github_upload/karsl/notebooks"

print("=== NOTEBOOK SEARCH ===")

found = []

for p in D.rglob("*.ipynb"):
    if "github_upload" in p.parts:
        continue

    if "karsl" in p.name.lower() or "label" in p.name.lower():
        found.append(p)
        print(p.relative_to(D))

print("\n=== SEARCH SUMMARY ===")
print("Matching notebooks:", len(found))
print("Label review notebook found:", any(
    "label" in p.name.lower() for p in found
))

In [ ]:
from pathlib import Path
import json

D = Path("/content/drive/MyDrive")
src = D / "Colab Notebooks/karsl label review.ipynb"
dst = D / "SIGNOVA/KArSL/github_upload/karsl/notebooks/karsl_label_review.ipynb"

with open(src, encoding="utf-8") as f:
    nb = json.load(f)

for cell in nb.get("cells", []):
    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(
    json.dumps(nb, ensure_ascii=False, indent=1),
    encoding="utf-8"
)

print("=== NOTEBOOK INTEGRATION ===")
print("Status: PREPARED")
print("Notebook:", dst.name)
print("Cells:", len(nb["cells"]))
print("Size MB:", round(dst.stat().st_size / 1048576, 2))

print("\n=== ALL STAGED NOTEBOOKS ===")
for p in sorted(dst.parent.glob("*.ipynb")):
    print(p.name, round(p.stat().st_size / 1048576, 2), "MB")